# Pelatihan model visi singkong di Kaggle (GPU)

Notebook ini menjalankan `scripts/train_vision.py` untuk melatih klasifikasi 5 kelas (CBB, CBSD, CGM, CMD, Healthy) dari `merged.csv`.

**Persiapan (panel kanan Kaggle):**
1. Session options → Accelerator: GPU (T4 / P100).
2. Session options → Internet: On (untuk mengunduh bobot ImageNet).
3. Add Input → dataset citra `tahsin/cassava-leaf-disease-merged` (berisi `merged.csv` dan folder `train`).
4. Add Input → dataset kode: unggah zip repositori sebagai Dataset baru (Kaggle mengekstraknya otomatis).

Jalankan sel secara berurutan. Sel uji cepat (langkah 5) mengukur waktu sebelum pelatihan penuh.

In [ ]:
import subprocess, sys
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    gpu = r.stdout.strip()
except FileNotFoundError:
    gpu = ""
print("GPU:", gpu or "TIDAK ADA (aktifkan Accelerator = GPU di Session options, lalu restart sesi)")
import torch; print("torch", torch.__version__, "| cuda tersedia:", torch.cuda.is_available())

In [ ]:
!pip install -q timm
import timm; print("timm", timm.__version__)

## 3. Deteksi kode dan data

Sel berikut menampilkan struktur `/kaggle/input`, mencari kode dan `merged.csv` (atau CSV lain yang memiliki kolom `image_id`),
melacak folder citra dari `image_id` pertama, lalu memeriksa 200 baris pertama memiliki file gambar. Bila ada yang tidak
cocok, sel berhenti dengan pesan yang menjelaskan bagian yang perlu diubah.

In [ ]:
import glob, os, sys
from pathlib import Path
import pandas as pd

KAGGLE_INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = Path(os.environ.get("KAGGLE_WORK", "/kaggle/working"))

# --- 1. struktur folder (kedalaman 3) untuk membantu diagnosis
def show_tree(root, max_depth=3):
    root = str(root)
    for dp, dn, fn in os.walk(root):
        depth = dp[len(root):].count(os.sep)
        if depth >= max_depth:
            dn[:] = []
            continue
        print("  " * depth + os.path.basename(dp) + f"/   ({len(fn)} file, {len(dn)} subfolder)")
        for f in sorted(fn)[:3]:
            print("  " * (depth + 1) + f)
show_tree(KAGGLE_INPUT)

# --- 2. kode
hits = glob.glob(f"{KAGGLE_INPUT}/**/scripts/train_vision.py", recursive=True)
assert hits, "Kode (scripts/train_vision.py) tidak ditemukan. Unggah zip repo sebagai Dataset lalu Add Input."
TRAIN_SCRIPT = Path(hits[0]); CODE_ROOT = TRAIN_SCRIPT.parent.parent
sys.path.insert(0, str(CODE_ROOT))
from cassava_es.data import resolve_image_path

# --- 3. CSV
ID_COL, LABEL_COL, SOURCE_COL = "image_id", "label", "source"   # ubah bila nama kolom berbeda
csvs = glob.glob(f"{KAGGLE_INPUT}/**/merged.csv", recursive=True) or glob.glob(f"{KAGGLE_INPUT}/**/*.csv", recursive=True)
CSV = None
for c in csvs:
    try:
        if ID_COL in pd.read_csv(c, nrows=1).columns:
            CSV = Path(c); break
    except Exception:
        pass
assert CSV is not None, f"Tidak ada CSV berkolom '{ID_COL}' di {KAGGLE_INPUT}. Sudah Add Input dataset citra? CSV yang ada: {csvs[:5]}"
df = pd.read_csv(CSV)
print("\ncsv  :", CSV, "| baris:", len(df), "| kolom:", list(df.columns))
for col in (ID_COL, LABEL_COL):
    assert col in df.columns, f"Kolom '{col}' tidak ada. Kolom tersedia: {list(df.columns)}. Ubah ID_COL/LABEL_COL di atas."
if SOURCE_COL not in df.columns:
    print(f"(kolom '{SOURCE_COL}' tidak ada; tidak masalah, hanya dipakai untuk laporan)")
print("contoh label:", df[LABEL_COL].value_counts().head(8).to_dict())

# --- 4. folder citra: lacak lokasi file pertama
sample = str(df[ID_COL].iloc[0])
found = glob.glob(f"{CSV.parent}/**/{glob.escape(sample)}", recursive=True) or glob.glob(f"{CSV.parent}/**/{glob.escape(sample)}.*", recursive=True)
assert found, f"File untuk image_id '{sample}' tidak ditemukan di bawah {CSV.parent}. Periksa struktur folder di atas dan isi IMG_DIR manual."
IMG_DIR = Path(found[0]).parent          # ubah manual bila perlu
n_check = min(200, len(df))
ok = sum(resolve_image_path(IMG_DIR, str(i), None) is not None for i in df[ID_COL].head(n_check))
print("citra:", IMG_DIR, f"| {ok}/{n_check} baris pertama punya file gambar")
assert ok >= 0.9 * n_check, "Kurang dari 90% baris pertama punya file di IMG_DIR (mungkin citra tersebar di beberapa subfolder). Periksa struktur folder."
COLS = f"--id-col {ID_COL} --label-col {LABEL_COL} --source-col {SOURCE_COL}"
print("\nKode :", CODE_ROOT)

## 4. Pengaturan

* `MODEL`: default `volo_d2_384` (model pada paper); alternatif yang lebih ringan: `tf_efficientnetv2_s`.
* Bila GPU kehabisan memori: kecilkan `BATCH_SIZE` atau `IMG_SIZE`.
* `VAL_SIZE`: `0.0` meniru paper (80/20, checkpoint epoch terakhir). `0.1` (disarankan) memisahkan 10 % data
  latih sebagai validasi, memilih epoch dengan macro F1 validasi tertinggi, dan menguji data uji satu kali.
* Citra ganda dapat diperiksa lebih dulu dengan `scripts/check_duplicates.py` (lihat README).

In [ ]:
MODEL      = "volo_d2_384"   # alternatif ringan: "tf_efficientnetv2_s"
IMG_SIZE   = 384
EPOCHS     = 10
BATCH_SIZE = 16
NUM_WORKERS = 4
VAL_SIZE    = 0.0            # 0.1 = pakai data validasi untuk memilih epoch

## 5. Uji cepat (±300 citra, 1 epoch): mengukur waktu dan memastikan tidak ada galat

In [ ]:
import time
t0 = time.time()
!python {TRAIN_SCRIPT} --csv {CSV} --img-dir {IMG_DIR} --out {WORK}/smoke/vision.pt \
    --model {MODEL} --img-size {IMG_SIZE} --epochs 1 --batch-size {BATCH_SIZE} \
    --num-workers {NUM_WORKERS} --limit 300 {COLS}
print(f"\nWaktu uji cepat: {time.time()-t0:.0f} dtk untuk ~240 citra latih. "
      "Perkirakan waktu 1 epoch penuh ≈ (jumlah citra latih / 240) × (waktu training di atas, tanpa unduh model).")

## 6. Pelatihan penuh

Tabel distribusi kelas pada keluaran dipakai untuk memeriksa pemetaan label (CMD seharusnya kelas terbanyak).
Checkpoint ditulis ulang tiap epoch di `/kaggle/working/vision/`; gunakan **Save Version** agar file tidak hilang saat sesi berakhir.

In [ ]:
!python {TRAIN_SCRIPT} --csv {CSV} --img-dir {IMG_DIR} --out {WORK}/vision/vision.pt \
    --model {MODEL} --img-size {IMG_SIZE} --epochs {EPOCHS} --batch-size {BATCH_SIZE} \
    --num-workers {NUM_WORKERS} --val-size {VAL_SIZE} {COLS}

## 7. Hasil pelatihan

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

OUT = WORK / "vision"
res = json.loads((OUT / "vision_metrics.json").read_text())
fm, hist = res["final_test_metrics"], res["meta"]["history"]
print(f"Akurasi uji: {fm['accuracy']:.4f} (n={fm['n']})   | paper VOLO-D2@384: 0.8964")
print(f"{'kelas':8s} {'presisi':>8s} {'recall':>8s} {'F1':>8s} {'n':>6s}")
for c, r in fm["per_class"].items():
    print(f"{c:8s} {r['precision']:8.3f} {r['recall']:8.3f} {r['f1']:8.3f} {r['support']:6d}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([h["epoch"] for h in hist], [h["train_loss"] for h in hist], marker="o", label="loss latih")
acc_key, acc_label = (("val_accuracy", "akurasi validasi") if "val_accuracy" in hist[0]
                      else ("test_accuracy", "akurasi uji"))
ax[0].plot([h["epoch"] for h in hist], [h[acc_key] for h in hist], marker="o", label=acc_label)
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].set_title("Riwayat pelatihan")
cm = np.array(fm["confusion_matrix"]); labels = fm["labels"]
ax[1].imshow(cm, cmap="Blues"); ax[1].set_title("Matriks konfusi (baris=benar)")
ax[1].set_xticks(range(5), labels, rotation=45); ax[1].set_yticks(range(5), labels)
for i in range(5):
    for j in range(5):
        ax[1].text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

## 8. Mengambil hasil

1. **Save Version → Save & Run All (Commit)**.
2. Pada tab **Output** versi tersebut, unduh `vision/vision.pt` dan `vision/vision_metrics.json`.
3. Letakkan `vision.pt` di `checkpoints/` pada repositori, jalankan `streamlit run app.py`, pada tab **Diagnosa** pilih *Foto daun*, lalu unggah foto
   (path checkpoint dapat diubah pada **Pengaturan lanjutan**).

In [ ]:
# Pemeriksaan: checkpoint dapat dimuat dan dipakai untuk inferensi (seperti di aplikasi)
import pandas as pd
from cassava_es.vision import VisionModel
vm = VisionModel.load(OUT / "vision.pt")
print(vm.describe())
split = pd.read_csv(OUT / "vision_split.csv")
img_id = split[split.split == "test"].iloc[0].image_id
from cassava_es.data import resolve_image_path
print(img_id, vm.predict_proba(resolve_image_path(IMG_DIR, img_id)))